In [1]:
# Gestion des chemins
from pathlib import Path

# Données et calculs
import pandas as pd
import numpy as np

# Dataviz
import matplotlib.pyplot as plt
%matplotlib inline

# Standardisation
from sklearn.preprocessing import StandardScaler

# Réduction de dimension
from sklearn.decomposition import PCA

# Séparation des données entrainement / validation
from sklearn.model_selection import train_test_split

# Modélisation
from sklearn.linear_model import LinearRegression
from sklearn.metrics import root_mean_squared_error


## Datasets

Lors de l'étape de feature engineering, on a obtenu des datasets de 169 variables en plus de l'index temporel. 

On s'est alors demandé s'il était pertinent de conserver les mesures collectées et calculées pour chacune des cinq communes identifiées à l'étape de clustering. Pour rappel : on a obtenu une trentaine de variables spécifiques à chaque commune.

On a donc calculé les sommes pondérées de ces variables pour en obtenir une version régionale. Les coefficients de pondérations ont été calculés par rapport à la part de la production d'énergie produite par chaque commune : une commune produisant plus d'énergie que sa voisine pèsera plus lourd que cette dernière dans le résultat final. Cette approche est similaire à celle utilisée dans l'article "Towards accurate forecasting of renewable energy: building datasets and benchmarking machine learning models for solar and wind power in France" (Eloi Lindas, Yannig Goude, Philippe Ciais) arXiv:2504.16100v1 14 avril 2025.

On a obtenu un deuxième jeu de datasets à 49 variables en plus de l'index temporel (169 - (5-1)*30 = 49).

Dans ce notebook, nous testerons les résultats des modèles obtenus sur le dataset à 169 variables et sur celui à 49 variables. 


In [2]:
# Chemins vers les datasets de travail
filepath_train_5pth = Path("../../data/train_5pts.csv")
filepath_train_region = Path("../../data/train_region.csv")
# On n'utilise pas les jeux de validation : ils serviront lors de la recherche des meilleurs paramètres
# On n'utilise pas les jeux de tests : ils serviront à la toute fin du projet

# Chemin vers les résultats de travail
filepath_output = Path("../../data//local_data/05_Modelisation/output/")

In [3]:
# Chargement des datasets
df_5pth = pd.read_csv(filepath_train_5pth, index_col='datetime_utc', parse_dates=True)
df_region = pd.read_csv(filepath_train_region, index_col='datetime_utc', parse_dates=True)


In [4]:
# Affichage
print("Dataset communes :")
display(df_5pth.head())
print("Taille du dataset :", df_5pth.shape)

print("\n Dataset régional :")
display(df_region.head())
print("Taille du dataset :", df_region.shape)

Dataset communes :


,bra_altitude,bra_azimuth,bra_bhi,bra_bni,bra_clear_sky_bhi,bra_clear_sky_bni,bra_clear_sky_dhi,bra_clear_sky_ghi,bra_dghi_dt,bra_dghi_lag_1,...,target_lag_3,tch_lag_1,tch_lag_2,tch_lag_3,tch_mean_2h,tch_mean_4h,tch_solaire,tch_std_2h,tch_std_4h,tco_solaire
datetime_utc,,,,,,,,,,,,,,,,,,,,,
2020-01-01 03:00:00+00:00,-44.186485,80.264541,0.0,0.0,0.0,0.0,0.0,0.0,0.0,0.0,...,0.0,0.0,0.0,0.0,0.0,0.0,0.0,0.0,0.0,0.0
2020-01-01 03:30:00+00:00,-38.785359,86.000566,0.0,0.0,0.0,0.0,0.0,0.0,0.0,0.0,...,0.0,0.0,0.0,0.0,0.0,0.0,0.0,0.0,0.0,0.0
2020-01-01 04:00:00+00:00,-33.346881,91.256074,0.0,0.0,0.0,0.0,0.0,0.0,0.0,0.0,...,0.0,0.0,0.0,0.0,0.0,0.0,0.0,0.0,0.0,0.0
2020-01-01 04:30:00+00:00,-27.918331,96.207025,0.0,0.0,0.0,0.0,0.0,0.0,0.0,0.0,...,0.0,0.0,0.0,0.0,0.0,0.0,0.0,0.0,0.0,0.0
2020-01-01 05:00:00+00:00,-22.539213,100.984524,0.0,0.0,0.0,0.0,0.0,0.0,0.0,0.0,...,0.0,0.0,0.0,0.0,0.0,0.0,0.0,0.0,0.0,0.0


Taille du dataset : (70122, 169)

 Dataset régional :


,consommation,cos_doy,cos_hour,sin_doy,sin_hour,solaire,target,target_lag_1,target_lag_2,target_lag_3,...,region_ghi_mean_4h,region_ghi_std_2h,region_ghi_std_4h,region_humidite,region_nebulosite,region_nebulosite_lag_1,region_nebulosite_lag_2,region_temperature,region_toa,region_vitesse_vent
datetime_utc,,,,,,,,,,,,,,,,,,,,,
2020-01-01 03:00:00+00:00,5332.0,1.0,0.707107,0.0,0.707107,0.0,0.0,0.0,0.0,0.0,...,0.0,0.0,0.0,73.3023,32.16660,19.95190,7.7372,3.36130,0.0,0.96980
2020-01-01 03:30:00+00:00,5219.0,1.0,0.608761,0.0,0.793353,0.0,0.0,0.0,0.0,0.0,...,0.0,0.0,0.0,72.5645,32.21870,32.16660,19.9519,3.26355,0.0,0.97035
2020-01-01 04:00:00+00:00,5157.0,1.0,0.500000,0.0,0.866025,0.0,0.0,0.0,0.0,0.0,...,0.0,0.0,0.0,71.8267,32.27080,32.21870,32.1666,3.16580,0.0,0.97090
2020-01-01 04:30:00+00:00,5161.0,1.0,0.382683,0.0,0.923880,0.0,0.0,0.0,0.0,0.0,...,0.0,0.0,0.0,70.5524,32.58025,32.27080,32.2187,3.22595,0.0,0.98650
2020-01-01 05:00:00+00:00,5108.0,1.0,0.258819,0.0,0.965926,0.0,0.0,0.0,0.0,0.0,...,0.0,0.0,0.0,69.2781,32.88970,32.58025,32.2708,3.28610,0.0,1.00210


Taille du dataset : (70122, 49)


In [5]:
# Informations
print("Dataset communes :")
display(df_5pth.describe())
print(df_5pth.info())

print("\n Dataset régional :")
display(df_region.describe())
print(df_region.info())

Dataset communes :


,bra_altitude,bra_azimuth,bra_bhi,bra_bni,bra_clear_sky_bhi,bra_clear_sky_bni,bra_clear_sky_dhi,bra_clear_sky_ghi,bra_dghi_dt,bra_dghi_lag_1,...,target_lag_3,tch_lag_1,tch_lag_2,tch_lag_3,tch_mean_2h,tch_mean_4h,tch_solaire,tch_std_2h,tch_std_4h,tco_solaire
count,70122.000000,70122.000000,70122.000000,70122.000000,70122.000000,70122.000000,70122.000000,70122.000000,7.012200e+04,7.012200e+04,...,7.012200e+04,70122.000000,70122.000000,70122.000000,70122.000000,70122.000000,70122.000000,70122.000000,70122.000000,70122.000000
mean,0.343840,180.494337,65.150891,120.005040,88.443753,163.758081,24.261875,112.705628,-1.459145e-17,2.107654e-17,...,5.066475e-20,17.028103,17.028103,17.028103,17.028103,17.028103,17.028103,3.170734,6.063037,6.254048
std,34.371064,101.288240,109.741005,168.194965,123.722534,185.599377,31.123548,150.009322,2.723012e+01,2.723012e+01,...,4.112778e+00,24.159588,24.159588,24.159588,23.745065,22.588871,24.159588,4.052541,6.867132,9.114173
min,-69.947186,0.028904,0.000000,0.000000,0.000000,0.000000,0.000000,0.000000,-3.078535e+02,-3.078535e+02,...,-1.685000e+01,0.000000,0.000000,0.000000,0.000000,0.000000,0.000000,0.000000,0.000000,0.000000
25%,-25.943830,90.319587,0.000000,0.000000,0.000000,0.000000,0.000000,0.000000,-1.849575e+00,-1.849575e+00,...,-4.100000e-01,0.000000,0.000000,0.000000,0.000000,0.000000,0.000000,0.000000,0.000000,0.000000
50%,0.569989,180.058688,0.000000,0.000000,0.326450,7.832500,1.912550,2.344450,0.000000e+00,0.000000e+00,...,0.000000e+00,0.210000,0.210000,0.210000,1.177500,3.554375,0.210000,1.016973,3.348617,0.080000
75%,26.610932,270.809256,96.955475,266.360400,166.595350,366.333300,44.488025,215.652425,2.349800e+00,2.349800e+00,...,2.000000e-01,32.730000,32.730000,32.730000,32.081250,30.959375,32.730000,5.746050,11.107516,11.500000
max,69.905117,359.996740,462.752600,514.454900,462.752600,514.454900,226.935600,501.771600,3.105739e+02,3.105739e+02,...,2.497000e+01,91.350000,91.350000,91.350000,90.900000,88.960000,91.350000,19.723125,26.791862,42.860000


<class 'pandas.core.frame.DataFrame'>
DatetimeIndex: 70122 entries, 2020-01-01 03:00:00+00:00 to 2023-12-31 23:30:00+00:00
Columns: 169 entries, bra_altitude to tco_solaire
dtypes: float64(169)
memory usage: 90.9 MB
None

 Dataset régional :


,consommation,cos_doy,cos_hour,sin_doy,sin_hour,solaire,target,target_lag_1,target_lag_2,target_lag_3,...,region_ghi_mean_4h,region_ghi_std_2h,region_ghi_std_4h,region_humidite,region_nebulosite,region_nebulosite_lag_1,region_nebulosite_lag_2,region_temperature,region_toa,region_vitesse_vent
count,70122.000000,70122.000000,7.012200e+04,7.012200e+04,70122.000000,70122.000000,7.012200e+04,7.012200e+04,7.012200e+04,7.012200e+04,...,70122.000000,70122.000000,70122.000000,70122.000000,70122.000000,70122.000000,70122.000000,70122.000000,70122.000000,70122.000000
mean,4523.076467,-0.000086,-7.901413e-05,-2.594035e-17,-0.000027,283.456561,1.519943e-18,1.013295e-19,-6.586418e-19,5.066475e-20,...,94.354627,19.343497,34.772208,68.465636,50.066062,50.065091,50.063999,14.225347,156.253333,2.644384
std,872.331520,0.707082,7.070901e-01,7.071421e-01,0.707133,410.208402,4.112778e+00,4.112778e+00,4.112778e+00,4.112778e+00,...,126.506116,22.009468,36.147965,20.439658,34.207745,34.207633,34.207763,8.553171,197.657673,1.478891
min,2620.000000,-1.000000,-1.000000e+00,-9.999907e-01,-1.000000,0.000000,-1.685000e+01,-1.685000e+01,-1.685000e+01,-1.685000e+01,...,0.000000,0.000000,0.000000,15.017300,0.054200,0.054200,0.054200,-4.733200,0.000000,0.164400
25%,3877.000000,-0.708627,-7.071068e-01,-7.055836e-01,-0.707107,0.000000,-4.100000e-01,-4.100000e-01,-4.100000e-01,-4.100000e-01,...,0.000000,0.000000,0.000000,52.957462,17.275950,17.275950,17.273800,7.376975,0.000000,1.434600
50%,4409.000000,0.004304,-1.836970e-16,0.000000e+00,0.000000,3.000000,0.000000e+00,0.000000e+00,0.000000e+00,0.000000e+00,...,24.057478,9.868720,25.185840,71.782575,45.538100,45.536025,45.534325,13.382850,8.363064,2.394200
75%,5091.000000,0.704066,7.071068e-01,7.055836e-01,0.707107,529.000000,2.000000e-01,2.000000e-01,2.000000e-01,2.000000e-01,...,161.839967,37.622776,63.847998,85.605712,85.236875,85.236187,85.236187,20.298350,304.119177,3.500700
max,8044.000000,1.000000,1.000000e+00,9.999907e-01,1.000000,1784.000000,2.497000e+01,2.497000e+01,2.497000e+01,2.497000e+01,...,485.445047,144.790931,173.155164,100.000000,100.000000,100.000000,100.000000,39.457900,617.540712,10.552600


<class 'pandas.core.frame.DataFrame'>
DatetimeIndex: 70122 entries, 2020-01-01 03:00:00+00:00 to 2023-12-31 23:30:00+00:00
Data columns (total 49 columns):
 #   Column                    Non-Null Count  Dtype  
---  ------                    --------------  -----  
 0   consommation              70122 non-null  float64
 1   cos_doy                   70122 non-null  float64
 2   cos_hour                  70122 non-null  float64
 3   sin_doy                   70122 non-null  float64
 4   sin_hour                  70122 non-null  float64
 5   solaire                   70122 non-null  float64
 6   target                    70122 non-null  float64
 7   target_lag_1              70122 non-null  float64
 8   target_lag_2              70122 non-null  float64
 9   target_lag_3              70122 non-null  float64
 10  tch_lag_1                 70122 non-null  float64
 11  tch_lag_2                 70122 non-null  float64
 12  tch_lag_3                 70122 non-null  float64
 13  tch_mean_2h   

## Préparation à la modélisation

On a vu que toutes nos colonnes sont numériques. 

On va maintenant les diviser en jeux d'entrainement et en jeux de test. 

Pour rappel : on travaille ici sur les années 2020 à 2023 (datasets dit d'entrainement, mais qu'on va ici rediviser en train et test) : on introduira l'année 2024 dans le notebook suivant et l'année 2025 ne sera utilisée que pour évaluer le comportement du modèle final retenu en production).

Après cette division train / test, on va normaliser les données en se basant sur le train, de manière à permettre la comparaison du plus grand nombre de modélisations possible.


## Protocole de recherche de modèle

Dans ce notebook, on va rechercher quels types de modèles obtiennent les meilleurs résultats sur nos jeux de données d'entrainement.

Dans un prochain notebook, on fera une recherche des meilleurs hyperparamètres pour les modèles retenus, en s'appuyant également sur les datasets de validation.

Enfin, à la toute fin du projet, on évaluera les performances du modèle retenu sur les datasets de test.